# RAINSTORM supervised workflow with STORM

Notebook tutorial for loading the supervised example data, keeping TensorFlow isolated behind the supervised extra, and running a lightweight STORM proxy over the same position table.

In [ ]:
from pathlib import Path
import sys
from tempfile import TemporaryDirectory


def find_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "packages" / "rainstorm-thesis" / "src" / "rainstorm_thesis").is_dir():
            return candidate
    raise RuntimeError("RAINSTORM project root not found")


PROJECT_ROOT = find_root(Path.cwd())
THESIS_SRC = PROJECT_ROOT / "packages" / "rainstorm-thesis" / "src"
for path in (PROJECT_ROOT, THESIS_SRC):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

PROJECT_ROOT

In [ ]:
from storm import FileArtifactStore, ModelRegistry, Study, StudySpec
from rainstorm_thesis import (
    PoseDatasetConfig,
    RainstormSupervisedProxyModel,
    build_pose_dataset_loader,
    build_supervised_study_spec,
    example_colabels_file,
    load_pose_table,
)

colabels_path = example_colabels_file()
pose_table = load_pose_table(colabels_path, max_frames=2_000)
pose_table.filter(regex="Labeler|_x|_y").head()

In [ ]:
config = PoseDatasetConfig(pose_path=colabels_path, label_path=colabels_path, max_frames=2_000)
data_ref = config.data_ref(identifier="nor-supervised-colabels")
data_loader = build_pose_dataset_loader(config)
dataset = data_loader(data_ref)

dataset.inputs.shape, dataset.targets.head().tolist()

In [ ]:
model_path = PROJECT_ROOT / "examples" / "models" / "trained_models" / "example_simple.keras"

models = ModelRegistry()
models.register("supervised", RainstormSupervisedProxyModel)

spec: StudySpec = build_supervised_study_spec(
    study_id="rainstorm-supervised-storm-demo",
    data=data_ref,
    model_path=str(model_path),
    thresholds=(0.35, 0.5, 0.65),
)

spec.to_dict()

In [ ]:
workspace = TemporaryDirectory(prefix="rainstorm-supervised-storm-")
study = Study(
    spec,
    data_loader=data_loader,
    models=models,
    artifacts=FileArtifactStore(Path(workspace.name) / "artifacts"),
)
results = study.run()

[(result.run_id, result.load_output().metadata) for result in results]

In [ ]:
workspace.cleanup()